# Seasonality Modeling of Timeseries Data

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

In [16]:
#corrected data with means for each group of 5 animals
wnt_data_corrected = pd.read_csv("wnt_corrected.csv")
#unique indexes for sanity checking
wnt_data_corrected.index = wnt_data_corrected["gene"] + wnt_data_corrected["Time.Point"].apply(lambda x: x.replace("Day ", "D."))
#get relative error bar positions values as offsets of means
wnt_data_corrected["lower_err"] = abs(wnt_data_corrected["lower"] - wnt_data_corrected["mean"])
wnt_data_corrected["upper_err"] = abs(wnt_data_corrected["upper"] - wnt_data_corrected["mean"])
del wnt_data_corrected["Unnamed: 0"]

wnt_data_corrected

,gene,Time.Point,mean,sd,lower,upper,lower_err,upper_err
wnt9D.1,wnt9,Day 1,0.138649,0.211341,-0.200350,0.473402,0.338999,0.334753
wnt9D.14,wnt9,Day 14,0.257552,0.211944,-0.083801,0.600797,0.341353,0.343245
wnt9D.21,wnt9,Day 21,-0.071788,0.213450,-0.428650,0.281111,0.356861,0.352899
wnt9D.3,wnt9,Day 3,0.771689,0.215710,0.408644,1.121518,0.363045,0.349830
wnt9D.5,wnt9,Day 5,0.462917,0.214714,0.120594,0.832663,0.342323,0.369746
wnt9D.7,wnt9,Day 7,0.302412,0.216310,-0.043544,0.655283,0.345956,0.352871
wnt6D.1,wnt6,Day 1,-0.620026,0.367346,-1.195549,-0.041525,0.575524,0.578501
wnt6D.14,wnt6,Day 14,0.958533,0.368885,0.362158,1.553291,0.596375,0.594759
wnt6D.21,wnt6,Day 21,0.770463,0.371845,0.153110,1.382216,0.617353,0.611753
wnt6D.3,wnt6,Day 3,1.381159,0.361248,0.788282,1.963123,0.592877,0.581964


In [17]:
## Rename Time.Point to time_point and derive days as numbers
wnt_data_corrected['time_point'] = wnt_data_corrected['Time.Point'].str.replace("Day ", "", regex=False)
wnt_data_corrected['time_point'] = wnt_data_corrected['time_point'].astype(int)

## Turn time_point into a date 1/X/2025
wnt_data_corrected['date'] = pd.to_datetime("2024-12-31") + pd.to_timedelta(wnt_data_corrected['time_point'], unit='D')

## Drop Time.Point columns
wnt_data_corrected = wnt_data_corrected.drop(columns=['Time.Point', 'time_point'])

wnt_data_corrected.head()

,gene,mean,sd,lower,upper,lower_err,upper_err,date
wnt9D.1,wnt9,0.138649,0.211341,-0.200350,0.473402,0.338999,0.334753,2025-01-01
wnt9D.14,wnt9,0.257552,0.211944,-0.083801,0.600797,0.341353,0.343245,2025-01-14
wnt9D.21,wnt9,-0.071788,0.213450,-0.428650,0.281111,0.356861,0.352899,2025-01-21
wnt9D.3,wnt9,0.771689,0.215710,0.408644,1.121518,0.363045,0.349830,2025-01-03
wnt9D.5,wnt9,0.462917,0.214714,0.120594,0.832663,0.342323,0.369746,2025-01-05


In [18]:
wnt_data_corrected = wnt_data_corrected.set_index('date')

wnt_data_corrected.index = pd.PeriodIndex(wnt_data_corrected.index, freq='D')

wnt_data_corrected

,gene,mean,sd,lower,upper,lower_err,upper_err
date,,,,,,,
2025-01-01,wnt9,0.138649,0.211341,-0.200350,0.473402,0.338999,0.334753
2025-01-14,wnt9,0.257552,0.211944,-0.083801,0.600797,0.341353,0.343245
2025-01-21,wnt9,-0.071788,0.213450,-0.428650,0.281111,0.356861,0.352899
2025-01-03,wnt9,0.771689,0.215710,0.408644,1.121518,0.363045,0.349830
2025-01-05,wnt9,0.462917,0.214714,0.120594,0.832663,0.342323,0.369746
2025-01-07,wnt9,0.302412,0.216310,-0.043544,0.655283,0.345956,0.352871
2025-01-01,wnt6,-0.620026,0.367346,-1.195549,-0.041525,0.575524,0.578501
2025-01-14,wnt6,0.958533,0.368885,0.362158,1.553291,0.596375,0.594759
2025-01-21,wnt6,0.770463,0.371845,0.153110,1.382216,0.617353,0.611753


In [13]:
def get_gene_data(gene_name:str) -> pd.DataFrame:
    gene_data = wnt_data_corrected[wnt_data_corrected['gene'] == gene_name].copy()[['mean']]
    gene_data.index = pd.PeriodIndex(gene_data.index, freq='D')
    return gene_data

# wnt9_data = wnt_data_pvt[wnt_data_pvt['gene'] == 'wnt9'].copy()[['count']]
# wnt9_data = wnt_data_pvt[wnt_data_pvt['gene'] == 'wnt9'].copy()[['count']]

# wnt9_data

## Time Series Decomposition

Time series decomposition divides the data into its trend, seasonal, and residual components.

In [19]:
## Perform time series decomposition
## Genes: ['wnt9', 'tubb', 'ubc', 'tuba', 'wnt3a', 'wnt6', 'wnt4', 'wnt5b', 'wnt10b', 'wnt16']

wnt3a_decomp = seasonal_decompose(get_gene_data('wnt3a'), model='stl', period=1)
wnt6_decomp = seasonal_decompose(get_gene_data('wnt6'), model='stl', period=1)
wnt4_decomp = seasonal_decompose(get_gene_data('wnt4'), model='stl', period=1)
wnt5b_decomp = seasonal_decompose(get_gene_data('wnt5b'), model='stl', period=1)
wnt9_decomp = seasonal_decompose(get_gene_data('wnt9'), model='stl', period=1)
wnt10b_decomp = seasonal_decompose(get_gene_data('wnt10b'), model='stl', period=1)
wnt16_decomp = seasonal_decompose(get_gene_data('wnt16'), model='stl', period=1)
# tubb_decomp = seasonal_decompose(get_gene_data('tubb'), model='stl', period=1)
# ubc_decomp = seasonal_decompose(get_gene_data('ubc'), model='stl', period=1)
# tuba_decomp = seasonal_decompose(get_gene_data('tuba'), model='stl', period=1)


In [20]:
wnt_decomp_df = pd.DataFrame({
    "resid_wnt9": wnt9_decomp.resid,
    "seasonal_wnt9": wnt9_decomp.seasonal,
    "trend_wnt9": wnt9_decomp.trend,
    "weights_wnt9": wnt9_decomp.weights,
    # "resid_tubb": tubb_decomp.resid,
    # "seasonal_tubb": tubb_decomp.seasonal,
    # "trend_tubb": tubb_decomp.trend,
    # "weights_tubb": tubb_decomp.weights,
    # "resid_ubc": ubc_decomp.resid,
    # "seasonal_ubc": ubc_decomp.seasonal,
    # "trend_ubc": ubc_decomp.trend,
    # "weights_ubc": ubc_decomp.weights,
    # "resid_tuba": tuba_decomp.resid,
    # "seasonal_tuba": tuba_decomp.seasonal,
    # "trend_tuba": tuba_decomp.trend,
    # "weights_tuba": tuba_decomp.weights,
    "resid_wnt3a": wnt3a_decomp.resid,
    "seasonal_wnt3a": wnt3a_decomp.seasonal,
    "trend_wnt3a": wnt3a_decomp.trend,
    "weights_wnt3a": wnt3a_decomp.weights,
    "resid_wnt6": wnt6_decomp.resid,
    "seasonal_wnt6": wnt6_decomp.seasonal,
    "trend_wnt6": wnt6_decomp.trend,
    "weights_wnt6": wnt6_decomp.weights,
    "resid_wnt4": wnt4_decomp.resid,
    "seasonal_wnt4": wnt4_decomp.seasonal,
    "trend_wnt4": wnt4_decomp.trend,
    "weights_wnt4": wnt4_decomp.weights,
    "resid_wnt5b": wnt5b_decomp.resid,
    "seasonal_wnt5b": wnt5b_decomp.seasonal,
    "trend_wnt5b": wnt5b_decomp.trend,
    "weights_wnt5b": wnt5b_decomp.weights,
    "resid_wnt10b": wnt10b_decomp.resid,
    "seasonal_wnt10b": wnt10b_decomp.seasonal,
    "trend_wnt10b": wnt10b_decomp.trend,
    "weights_wnt10b": wnt10b_decomp.weights,
    "resid_wnt16": wnt16_decomp.resid,
    "seasonal_wnt16": wnt16_decomp.seasonal,
    "trend_wnt16": wnt16_decomp.trend,
    "weights_wnt16": wnt16_decomp.weights,
})

## Rename index to 'date' for consistency
wnt_decomp_df.index.name = 'date'
## Turn index into a column
wnt_decomp_df.reset_index(inplace=True)

wnt_decomp_df.head()

,date,resid_wnt9,seasonal_wnt9,trend_wnt9,weights_wnt9,resid_wnt3a,seasonal_wnt3a,trend_wnt3a,weights_wnt3a,resid_wnt6,...,trend_wnt5b,weights_wnt5b,resid_wnt10b,seasonal_wnt10b,trend_wnt10b,weights_wnt10b,resid_wnt16,seasonal_wnt16,trend_wnt16,weights_wnt16
0,2025-01-01,0.0,0.0,0.138649,1.0,0.0,0.0,0.202602,1.0,0.0,...,-1.072670,1.0,0.0,0.0,-0.556617,1.0,0.0,0.0,-0.457546,1.0
1,2025-01-14,0.0,0.0,0.257552,1.0,0.0,0.0,-0.268129,1.0,0.0,...,1.127846,1.0,0.0,0.0,0.989542,1.0,0.0,0.0,0.387129,1.0
2,2025-01-21,0.0,0.0,-0.071788,1.0,0.0,0.0,-0.680927,1.0,0.0,...,0.879089,1.0,0.0,0.0,0.748339,1.0,0.0,0.0,0.248367,1.0
3,2025-01-03,0.0,0.0,0.771689,1.0,0.0,0.0,0.442483,1.0,0.0,...,0.168831,1.0,0.0,0.0,0.726185,1.0,0.0,0.0,0.732665,1.0
4,2025-01-05,0.0,0.0,0.462917,1.0,0.0,0.0,-0.137683,1.0,0.0,...,0.836197,1.0,0.0,0.0,0.696268,1.0,0.0,0.0,0.487854,1.0


In [21]:
wnt_decomp_df_long = pd.wide_to_long(wnt_decomp_df, stubnames=['resid', 'seasonal', 'trend', 'weights'], i=['date'], j='set', sep='_', suffix=r'\w+')

wnt_decomp_df_long.head()

,,resid,seasonal,trend,weights
date,set,,,,
2025-01-01,wnt9,0.0,0.0,0.138649,1.0
2025-01-14,wnt9,0.0,0.0,0.257552,1.0
2025-01-21,wnt9,0.0,0.0,-0.071788,1.0
2025-01-03,wnt9,0.0,0.0,0.771689,1.0
2025-01-05,wnt9,0.0,0.0,0.462917,1.0


In [22]:
wnt_decomp_df_long.to_csv("wnt_corrected_decomp_df_long.csv")